# Jeffi Products — category classification with micrograd

Take the autograd engine from `micrograd_solo.py` and train a real classifier on **live product data** from the jeffi_replica DB:

- **Features:** `length_cm`, `breadth_cm`, `height_cm`, `weight_grams` (4 numeric dims)
- **Target:** product category (top-5 by count → 5-class classification)
- **Rows:** ~860 products after filtering to top-5 categories with all 4 dims present

## Why this is interesting

Screws, Bolts, Nuts, and Drill Bits are mechanically distinct objects.  
A screw is small and light; a V-belt is long and flat; a drill bit is thin and dense.  
The hypothesis: **physical dimensions alone carry enough signal to separate these categories**.  
If the micrograd MLP can learn that, the autograd engine works on a real industrial ML task.

## Same constraints as Iris

- Pure scalar autograd — no tensors, no batching
- Full-batch SGD
- Softmax + cross-entropy (same loss as `iris_with_micrograd.ipynb`)
- Linear output layer (no tanh on logits)

In [ ]:
import math
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import psycopg2
import psycopg2.extras
from dotenv import load_dotenv
import os

# Find micrograd_solo.py regardless of CWD
NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, NOTEBOOK_DIR / 'experiments' / '01_micrograd']:
    if (candidate / 'micrograd_solo.py').exists():
        sys.path.insert(0, str(candidate))
        break
from micrograd_solo import Value

# Load DB creds from experiment 04 .env
_repo = Path.cwd()
for candidate in [_repo, _repo.parent, _repo.parent.parent]:
    env_path = candidate / 'experiments' / '04_jeffi_descgen' / '.env'
    if env_path.exists():
        load_dotenv(env_path)
        break

random.seed(42)
np.random.seed(42)
print('imports ok')
print('Value:', Value)

## Data — pull from jeffi_replica

Filter to top-5 categories by product count, require all 4 physical dimensions to be present.

In [ ]:
conn = psycopg2.connect(
    host=os.getenv('JEFFI_DB_HOST'),
    port=os.getenv('JEFFI_DB_PORT'),
    dbname=os.getenv('JEFFI_DB_NAME'),
    user=os.getenv('JEFFI_DB_USER'),
    password=os.getenv('JEFFI_DB_PASSWORD'),
)
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Top-5 categories with all 4 dims present
cur.execute("""
    SELECT c.name as category
    FROM products p
    JOIN categories c ON p.category_id = c.id
    WHERE length(p.description) >= 80
      AND p.length_cm IS NOT NULL AND p.breadth_cm IS NOT NULL
      AND p.height_cm IS NOT NULL AND p.weight_grams IS NOT NULL
    GROUP BY c.name
    ORDER BY COUNT(*) DESC
    LIMIT 5
""")
TOP5 = [r['category'] for r in cur.fetchall()]
print('Top-5 categories:', TOP5)

cur.execute("""
    SELECT
        p.length_cm::float   AS length_cm,
        p.breadth_cm::float  AS breadth_cm,
        p.height_cm::float   AS height_cm,
        p.weight_grams::float AS weight_grams,
        c.name               AS category
    FROM products p
    JOIN categories c ON p.category_id = c.id
    WHERE length(p.description) >= 80
      AND p.length_cm IS NOT NULL AND p.breadth_cm IS NOT NULL
      AND p.height_cm IS NOT NULL AND p.weight_grams IS NOT NULL
      AND c.name = ANY(%s)
""", (TOP5,))
rows = cur.fetchall()
conn.close()

print(f'Fetched {len(rows)} rows')

# Category → int label
CAT2IDX = {cat: i for i, cat in enumerate(TOP5)}
print('Label map:', CAT2IDX)

X_raw = np.array([[r['length_cm'], r['breadth_cm'], r['height_cm'], r['weight_grams']] for r in rows])
y_all = np.array([CAT2IDX[r['category']] for r in rows])
print(f'X shape: {X_raw.shape},  y shape: {y_all.shape}')
print(f'Class counts: { {cat: int((y_all==i).sum()) for cat,i in CAT2IDX.items()} }')

## Preprocessing

StandardScaler (zero mean, unit variance) — same reason as Iris. Raw `weight_grams` runs 1–5000; raw `length_cm` runs 0.5–300. Without scaling, the weight dimension dominates the dot product and the other features are invisible to the network.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_all, test_size=0.2, stratify=y_all, random_state=42
)
print(f'train: {len(X_train)}  test: {len(X_test)}')
print(f'feature means (scaled): {X_train.mean(axis=0).round(3)}')
print(f'feature stds  (scaled): {X_train.std(axis=0).round(3)}')

## EDA — physical dimensions by category

Before training: do the categories actually separate in feature space?

In [ ]:
import pandas as pd

df = pd.DataFrame(X_raw, columns=['length_cm', 'breadth_cm', 'height_cm', 'weight_grams'])
df['category'] = [TOP5[i] for i in y_all]

print('Median physical dims per category:')
print(df.groupby('category')[['length_cm', 'breadth_cm', 'height_cm', 'weight_grams']].median().round(1).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [('length_cm', 'weight_grams'), ('length_cm', 'height_cm'), ('breadth_cm', 'weight_grams')]
colors = ['#0969da', '#cf222e', '#1a7f37', '#9a6700', '#8250df']

for ax, (xcol, ycol) in zip(axes, pairs):
    for i, cat in enumerate(TOP5):
        mask = df['category'] == cat
        ax.scatter(df.loc[mask, xcol], df.loc[mask, ycol],
                   label=cat, alpha=0.45, s=18, color=colors[i])
    ax.set_xlabel(xcol); ax.set_ylabel(ycol)
    ax.set_title(f'{xcol} vs {ycol}')
    ax.grid(True, alpha=0.2)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.08), fontsize=9)
fig.suptitle('Jeffi products — raw feature space by category', y=1.02)
fig.tight_layout()
out = Path('runs') / 'jeffi_eda.png'
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=130, bbox_inches='tight')
print(f'wrote {out}')
plt.show()

## Model — 4 → 16 → 8 → 5, linear output

Same architecture pattern as `iris_with_micrograd.ipynb` — TanhLayers for hidden, LinearLayer for logits.

Bigger hidden layers (16, 8) than Iris (8) because we have more training examples and a harder separation.

In [ ]:
class LinearLayer:
    def __init__(self, nin, nout):
        self.w = [[Value(random.uniform(-1, 1)) for _ in range(nin)] for _ in range(nout)]
        self.b = [Value(0.0) for _ in range(nout)]

    def __call__(self, x):
        return [sum((wij * xj for wij, xj in zip(wi, x)), bi)
                for wi, bi in zip(self.w, self.b)]

    def parameters(self):
        return [p for wi in self.w for p in wi] + self.b


class TanhLayer(LinearLayer):
    def __call__(self, x):
        return [v.tanh() for v in super().__call__(x)]


class MLP:
    """Hidden TanhLayers + linear output layer (logits, no activation)."""
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = []
        for i in range(len(nouts) - 1):
            self.layers.append(TanhLayer(sizes[i], sizes[i + 1]))
        self.layers.append(LinearLayer(sizes[-2], sizes[-1]))

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for L in self.layers for p in L.parameters()]


def log_value(v):
    out = Value(math.log(max(v.data, 1e-15)), (v,), 'log')
    def _backward():
        v.grad += (1.0 / max(v.data, 1e-15)) * out.grad
    out._backward = _backward
    return out


def softmax(logits):
    m = max(l.data for l in logits)
    exps = [(l - m).exp() for l in logits]
    total = sum(e.data for e in exps)
    # Divide each exp by the scalar total (wraps total as a Value constant)
    return [e / total for e in exps]


def cross_entropy(logits, target_idx):
    probs = softmax(logits)
    return -log_value(probs[target_idx])


random.seed(42)
N_CLASSES = len(TOP5)
model = MLP(nin=4, nouts=[16, 8, N_CLASSES])
n_params = len(model.parameters())
print(f'MLP(4, [16, 8, {N_CLASSES}])  |  {n_params} parameters')
print(f'  4×16+16 = {4*16+16} hidden-1 params')
print(f'  16×8+8  = {16*8+8} hidden-2 params')
print(f'  8×{N_CLASSES}+{N_CLASSES}   = {8*N_CLASSES+N_CLASSES} output params')

## Train

Full-batch SGD. Wall time: several minutes (pure scalar autograd on ~700 examples × 200 ops × 100 steps).

The slow part isn't the math — it's Python overhead per scalar op. This is the same wall we hit on Iris, just bigger.

In [ ]:
def predict_class(model, x_row):
    logits = model([Value(float(v)) for v in x_row])
    return max(range(N_CLASSES), key=lambda i: logits[i].data)


def accuracy(model, X, y):
    return sum(predict_class(model, X[i]) == int(y[i]) for i in range(len(X))) / len(X)


lr = 0.01
steps = 300
history = []  # (step, mean_loss, train_acc, test_acc)
t0 = time.time()

for step in range(steps + 1):
    losses = []
    for i in range(len(X_train)):
        logits = model([Value(float(v)) for v in X_train[i]])
        losses.append(cross_entropy(logits, int(y_train[i])))

    total_loss = losses[0]
    for L in losses[1:]:
        total_loss = total_loss + L
    mean_loss = total_loss / len(losses)

    for p in model.parameters():
        p.grad = 0.0
    mean_loss.backward()
    for p in model.parameters():
        p.data -= lr * p.grad

    if step % 10 == 0:
        tr = accuracy(model, X_train, y_train)
        te = accuracy(model, X_test,  y_test)
        history.append((step, mean_loss.data, tr, te))
        elapsed = time.time() - t0
        print(f'step {step:3d}  loss {mean_loss.data:.4f}  train {tr:.3f}  test {te:.3f}  ({elapsed:.1f}s)')

print(f'\ntotal time: {time.time()-t0:.1f}s')

## Results

In [ ]:
steps_x  = [h[0] for h in history]
losses_y = [h[1] for h in history]
tr_y     = [h[2] for h in history]
te_y     = [h[3] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps_x, losses_y, marker='o', color='#cf222e', linewidth=1.6)
ax1.set_xlabel('step'); ax1.set_ylabel('mean cross-entropy')
ax1.set_title('Train loss — Jeffi products'); ax1.grid(True, alpha=0.3)
ax1.axhline(math.log(N_CLASSES), color='#6e7781', linestyle='--', linewidth=1,
            label=f'random baseline ln({N_CLASSES})={math.log(N_CLASSES):.2f}')
ax1.legend(fontsize=8)

ax2.plot(steps_x, tr_y, marker='o', label='train', color='#0969da', linewidth=1.6)
ax2.plot(steps_x, te_y, marker='s', label='test',  color='#cf222e', linewidth=1.6)
# Majority-class baseline: Screws are ~32% of top-5
majority_frac = max((y_all == i).sum() for i in range(N_CLASSES)) / len(y_all)
ax2.axhline(majority_frac, color='#6e7781', linestyle='--', linewidth=1,
            label=f'majority baseline {majority_frac:.2f}')
ax2.set_xlabel('step'); ax2.set_ylabel('accuracy')
ax2.set_title('Accuracy — Jeffi products'); ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3); ax2.legend(fontsize=8)

fig.tight_layout()
out = Path('runs') / 'jeffi_loss.png'
fig.savefig(out, dpi=130)
print(f'wrote {out}')
plt.show()

print(f'\nfinal train acc: {tr_y[-1]:.3f}')
print(f'final test  acc: {te_y[-1]:.3f}')
print(f'majority baseline: {majority_frac:.3f}')
if te_y[-1] > majority_frac + 0.10:
    print('Model beats majority baseline by >10 pp — dimensions carry real signal.')
else:
    print('Model near baseline — physical dims alone may not cleanly separate these categories.')

## Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred_test = [predict_class(model, X_test[i]) for i in range(len(X_test))]
cm = confusion_matrix(y_test, y_pred_test)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=TOP5)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion matrix — test set')
plt.xticks(rotation=30, ha='right')
fig.tight_layout()
out = Path('runs') / 'jeffi_confusion.png'
fig.savefig(out, dpi=130, bbox_inches='tight')
print(f'wrote {out}')
plt.show()

## Reflection

**What this proves:**  
Same `Value`, same `backward()`, same SGD — now running on live e-commerce product data from a Postgres DB instead of a toy 4-point dataset or sklearn Iris.

**What to watch:**  
- If test acc >> majority baseline (~0.30): physical dimensions do separate product categories meaningfully  
- If test acc ≈ baseline: Screws vs Bolts vs Nuts overlap heavily in dimension space (they're all small metal fasteners)

**The cost:**  
~700 examples × ~200 scalar ops × 100 steps ≈ 14M Python ops. That's why the next step is tensors — same math, 100× faster by operating on arrays instead of scalars one at a time.

**One line for `NOTES.md`:** What did the confusion matrix tell you about which categories are hardest to separate and why?